# 🕒 Cybercrime Clock: Discovering Emerging Cybercrime Patterns Across Indian Cities

**Course:** Data Science Lab  |  **Type:** Mini Project (Assignment 5)  
**Dataset:** [Cybersecurity Incidents in India (Kaggle)](https://www.kaggle.com/datasets/saisimha203/cybersecurity-cases-india), file `cybersecurity_cases_india_combined.csv` (the only dataset used)  
**Techniques:** Data cleaning → EDA → Hypothesis testing → Clustering (K-Means) → Classification (Logistic Regression, Decision Tree, Random Forest)

---

### 🎯 The story in one line
Not every Indian city faces the same cyber threats. Some are hit mostly by phishing, others by ransomware or fraud.  
**Can we (1) group cities by their cybercrime "fingerprint" and (2) predict whether an incident will cause a high financial loss?**

### 📋 About the data
Each row is **one cyber incident**:

| Column | Meaning |
|---|---|
| `Year`, `Day` | When it happened (2019–2024) |
| `Amount_Lost_INR` | Money lost in ₹ |
| `Incident_Type` | Phishing, Ransomware, Malware, etc. |
| `City` | Where it happened |
| `Category` | Who was targeted (Corporate, Health, Personal, etc.) |

> The dataset has **cities**, not states, so we analyse cities.

### 🗺️ Syllabus map
| Step | What we do | Syllabus unit |
|---|---|---|
| 1 | Load data with Pandas | 2.1 Data importing |
| 2 | Duplicates, missing values, outliers (Z-score) | 2.3 Cleaning & preprocessing |
| 3 | Summary statistics + 4 simple figures | 1.5 / 3.1 / 3.2 EDA |
| 4 | One-sample t-test + distribution | 4.1 Hypothesis testing & distributions |
| 5 | K-Means clustering of cities | Algorithmic modeling |
| 6 | Classification + evaluation | 4.2, 4.3, 4.4 |

## 📦 Step 0: Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
print("Libraries ready ✅")

## 📥 Step 1: Load the dataset
Keep the CSV in the same folder as this notebook.

In [ ]:
df = pd.read_csv("cybersecurity_cases_india_combined.csv")
print("Rows, Columns:", df.shape)
df.head()

In [ ]:
df.info()

## 🧹 Step 2: Data cleaning (Assignment 2.3)
We check for **duplicates, missing values and outliers**, and fix whatever we find.

In [ ]:
data = df.copy()

# 2.1 Duplicates
before = len(data)
data = data.drop_duplicates()
print(f"Duplicate rows removed: {before - len(data)}")

# 2.2 Missing values
print("\nMissing values per column:")
print(data.isnull().sum())

# (fill if any: text -> most common value, numbers -> median)
for c in ["Incident_Type", "City", "Category"]:
    data[c] = data[c].str.strip()
    data[c] = data[c].fillna(data[c].mode()[0])
data["Amount_Lost_INR"] = data["Amount_Lost_INR"].fillna(data["Amount_Lost_INR"].median())

In [ ]:
# 2.3 Outliers using the Z-score method (|z| > 3 = unusually extreme)
z = np.abs(stats.zscore(data["Amount_Lost_INR"]))
print(f"Outliers found (Z-score > 3): {(z > 3).sum()}")
data = data[z <= 3].copy()
print("Clean dataset shape:", data.shape)

## 📊 Step 3: Exploratory Data Analysis (Assignments 1.5, 3.1, 3.2)
First the numbers, then **4 simple figures**.

In [ ]:
print("Years covered :", data["Year"].min(), "to", data["Year"].max())
print("Cities        :", data["City"].nunique())
print("Incident types:", data["Incident_Type"].nunique())
print("Target sectors:", data["Category"].nunique())
print()
data[["Amount_Lost_INR"]].describe().round(0)

### Figure 1: Which cities lose the most money, and which attacks are most common?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

city_loss = data.groupby("City")["Amount_Lost_INR"].sum().sort_values() / 1e6
city_loss.plot.barh(ax=ax[0], color="steelblue")
ax[0].set_title("Total money lost by city")
ax[0].set_xlabel("Amount lost (₹ million)")

type_count = data["Incident_Type"].value_counts().sort_values()
type_count.plot.barh(ax=ax[1], color="indianred")
ax[1].set_title("Number of incidents by type")
ax[1].set_xlabel("Number of incidents")

plt.tight_layout()
plt.show()

### Figure 2: Is it changing over time? (Yearly trend) and how are losses spread? (Distribution)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

yearly = data.groupby("Year").agg(Incidents=("Amount_Lost_INR", "size"))
ax[0].plot(yearly.index, yearly["Incidents"], marker="o", color="darkgreen")
ax[0].set_xticks(yearly.index)
ax[0].set_title("Number of incidents per year")
ax[0].set_xlabel("Year"); ax[0].set_ylabel("Incidents")

sns.histplot(data["Amount_Lost_INR"], kde=True, stat="density", color="slateblue", ax=ax[1])
ax[1].set_title("Distribution of money lost per incident")
ax[1].set_xlabel("Amount lost (₹)")

plt.tight_layout()
plt.show()

### 🔥 "Emerging" threats: which incident types grew or shrank the most?
This is the heart of our title. We compare the **first year (2019)** with the **last year (2024)**.

In [ ]:
first_year, last_year = data["Year"].min(), data["Year"].max()
trend = data.pivot_table(index="Incident_Type", columns="Year", values="Amount_Lost_INR", aggfunc="size", fill_value=0)
growth = pd.DataFrame({str(first_year): trend[first_year], str(last_year): trend[last_year]})
growth["Change %"] = ((growth[str(last_year)] - growth[str(first_year)]) / growth[str(first_year)].replace(0, np.nan) * 100).round(1)
growth = growth.sort_values("Change %", ascending=False)
growth

## 🧪 Step 4: Hypothesis testing (Assignment 4.1)
**Question:** Is the average money lost per incident in the **latest year** really different from the **overall average**, or is the gap just chance?

- **H₀ (null):** Latest-year mean loss = overall mean loss  
- **H₁ (alternative):** Latest-year mean loss ≠ overall mean loss  
- **Test:** One-sample t-test, significance level α = 0.05

In [ ]:
population_mean = data["Amount_Lost_INR"].mean()
latest = data[data["Year"] == last_year]["Amount_Lost_INR"]

t_stat, p_value = stats.ttest_1samp(latest, population_mean)
print(f"Overall mean loss       : ₹{population_mean:,.0f}")
print(f"{last_year} mean loss          : ₹{latest.mean():,.0f}  (n = {len(latest)})")
print(f"t-statistic             : {t_stat:.3f}")
print(f"p-value                 : {p_value:.4f}")

if p_value < 0.05:
    print("\n✅ Reject H0: the latest year is significantly different from the overall average.")
else:
    print("\n❌ Fail to reject H0: no significant difference. The latest year looks like a normal year.")

## 🧩 Step 5: Clustering: grouping cities by their cybercrime "fingerprint"
For every city we calculate **what share of its incidents belongs to each incident type**, plus its average loss.  
**K-Means** then groups cities with similar fingerprints. No labels are needed, so this is *unsupervised* learning.

In [ ]:
# One row per city: share of each incident type (each row adds up to 100%)
counts = pd.crosstab(data["City"], data["Incident_Type"])
shares = counts.div(counts.sum(axis=1), axis=0)

features = shares.copy()
features["AvgLoss"] = data.groupby("City")["Amount_Lost_INR"].mean()
X_scaled = StandardScaler().fit_transform(features)

# Choose number of clusters (k) with the silhouette score (higher = better separated)
rows = []
for k in range(2, 6):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_scaled)
    rows.append((k, round(silhouette_score(X_scaled, labels), 3)))
sil = pd.DataFrame(rows, columns=["k (clusters)", "Silhouette score"])
print(sil.to_string(index=False))

best_k = int(sil.loc[sil["Silhouette score"].idxmax(), "k (clusters)"])
print("\nBest k =", best_k)

shares["Cluster"] = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit_predict(X_scaled)

### Figure 3: The fingerprint of each city (heatmap of incident-type shares)

In [ ]:
plt.figure(figsize=(11, 6))
order = shares.sort_values("Cluster").index          # cities in the same cluster sit together
sns.heatmap(shares.loc[order].drop(columns="Cluster") * 100, annot=True, fmt=".0f", cmap="YlOrRd",
            cbar_kws={"label": "% of city's incidents"})
plt.title("What kind of incidents does each city face? (%)")
plt.xlabel(""); plt.ylabel("")
plt.tight_layout()
plt.show()

### Figure 4: The city clusters on one map (PCA squeezes many features into 2 axes)

In [ ]:
coords = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
plot_df = pd.DataFrame(coords, columns=["PC1", "PC2"], index=shares.index)
plot_df["Cluster"] = shares["Cluster"].astype(str)

plt.figure(figsize=(9, 6))
sns.scatterplot(data=plot_df, x="PC1", y="PC2", hue="Cluster", palette="Set2", s=150)
for city, row in plot_df.iterrows():
    plt.text(row["PC1"] + 0.08, row["PC2"] + 0.08, city, fontsize=9)
plt.title("Cities grouped by cybercrime pattern (K-Means)")
plt.tight_layout()
plt.show()

In [ ]:
# Describe each cluster in plain words
summary = []
for c in sorted(shares["Cluster"].unique()):
    members = shares[shares["Cluster"] == c]
    top2 = members.drop(columns="Cluster").mean().nlargest(2)
    summary.append({
        "Cluster": c,
        "Cities": ", ".join(members.index),
        "Most common incidents": ", ".join(f"{t} ({v*100:.0f}%)" for t, v in top2.items()),
        "Avg loss (₹)": round(features.loc[members.index, "AvgLoss"].mean()),
    })
pd.DataFrame(summary)

## 🤖 Step 6: Classification: can we predict a **high-loss** incident? (Assignments 4.2, 4.3, 4.4)
**Target:** `HighLoss = 1` if the loss is at or above the median (about ₹2.2 lakh), otherwise `0`.  
**Inputs:** year, day, city, incident type and target sector.  
We compare three models, from simple to powerful: **Logistic Regression → Decision Tree → Random Forest**.

In [ ]:
data["HighLoss"] = (data["Amount_Lost_INR"] >= data["Amount_Lost_INR"].median()).astype(int)

X = pd.get_dummies(data[["Year", "Day", "City", "Incident_Type", "Category"]],
                   columns=["City", "Incident_Type", "Category"], dtype=int)
y = data["HighLoss"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("Training rows:", len(X_train), "| Testing rows:", len(X_test))

models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Decision Tree":       DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({"Model": name,
                    "Accuracy": round(accuracy_score(y_test, pred), 3),
                    "F1-score": round(f1_score(y_test, pred), 3)})
results = pd.DataFrame(results).sort_values("F1-score", ascending=False).reset_index(drop=True)
results

In [ ]:
# Detailed evaluation of the best model
best_name = results.loc[0, "Model"]
pred = models[best_name].predict(X_test)

print("Best model:", best_name, "\n")
cm = pd.DataFrame(confusion_matrix(y_test, pred),
                  index=["Actual Low", "Actual High"], columns=["Predicted Low", "Predicted High"])
print("Confusion matrix:\n", cm, "\n")
print(classification_report(y_test, pred, target_names=["Low loss", "High loss"]))

In [ ]:
# Which inputs mattered most? (Random Forest)
importance = pd.Series(models["Random Forest"].feature_importances_, index=X.columns).nlargest(5).round(3)
print("Top 5 most important inputs:")
importance

## 📝 Step 7: Key findings (auto-generated from the data)

In [ ]:
print("🔹 City with the highest total loss     :", data.groupby("City")["Amount_Lost_INR"].sum().idxmax())
print("🔹 Most common incident type            :", data["Incident_Type"].value_counts().idxmax())
print("🔹 Incident type with the highest avg loss:", data.groupby("Incident_Type")["Amount_Lost_INR"].mean().idxmax())
print("🔹 Fastest-growing incident type         :", growth["Change %"].idxmax(), f"({growth['Change %'].max():+.0f}%)")
print("🔹 Cities grouped into", best_k, "fingerprint clusters")
print("🔹 Hypothesis test (latest year vs. avg)  : p =", round(p_value, 4))
print("🔹 Best classifier                        :", best_name, "(accuracy =", results.loc[0, "Accuracy"], ")")

## ✅ Conclusion
- **Cities have different cybercrime fingerprints.** Clustering groups cities by the *type* of incidents they face, not only by how many.
- **Emerging threats** become visible when we compare the first and last year.
- **Classification:** if accuracy stays close to 50%, the model is only slightly better than a coin flip. That means city, incident type, sector and year **alone cannot predict** how large a loss will be. Reporting this honestly is a valid finding.

### ⚠️ Limitations (say this in your viva!)
- The Kaggle dataset is described as **simulated**, for research and education, so the results show the *method*, not real-world facts.
- It has 1,200 incidents across 10 cities, a small sample.
- There is no month or state information, only year and day.

### 🚀 Future scope
Add city population and internet-user data to compare incidents *per lakh people*, forecast next year's incidents with time-series models, and build a simple dashboard.

---
**Tools used:** Python, Pandas, NumPy, Matplotlib, Seaborn, SciPy, Scikit-learn